In [6]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils import extract_text
import re

In [7]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [8]:
file = ROOT / Path("examenes_quimica/2026_quimica/2026 - quimica - ordinaria titular titular quínica.pdf")

In [9]:
# Ya no hacen falta HEADER/INSTRUCTIONS hardcodeados: se eliminan por regex / por el split.

Diccionarios de sustitución de caracteres con `str.maketrans()`

In [ ]:
SUBSCRIPT = str.maketrans("0123456789+-–‒−", "₀₁₂₃₄₅₆₇₈₉₊₋₋₋₋")
SUPERSCRIPT = str.maketrans("0123456789+-–‒−", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁻⁻⁻")

### Claves de `page.chars` en pdfplumber

Cada elemento de `page.chars` es un diccionario con la información de un carácter.

**Identificación**

| Clave | Significado |
|---|---|
| `text` | El carácter |
| `object_type` | Siempre `'char'` |
| `page_number` | Número de página (empieza en 1) |

**Fuente**

| Clave | Significado |
|---|---|
| `fontname` | Nombre de la fuente; el prefijo (`BCDEEE+`) es el subconjunto incrustado, lo útil va detrás (`ArialNarrow-Bold` → negrita) |
| `size` | Tamaño en puntos. **Clave para detectar sub/superíndices** |
| `adv` | Avance horizontal tras el carácter |
| `upright` | `True` si el texto es horizontal |

**Posición** (en puntos; 1 pt = 1/72 de pulgada)

| Clave | Significado |
|---|---|
| `x0`, `x1` | Borde izquierdo y derecho, desde la izquierda de la página |
| `top`, `bottom` | Borde superior e inferior, **desde arriba** de la página |
| `y0`, `y1` | Lo mismo, pero **desde abajo** (sistema interno del PDF) |
| `doctop` | Como `top`, pero desde el inicio del documento |
| `width`, `height` | `x1 − x0` y `bottom − top` |
| `matrix` | Matriz de transformación: últimos dos valores = posición; primeros cuatro = escala/giro |

**Color**

| Clave | Significado |
|---|---|
| `non_stroking_color` | Color del texto (`(0.0,)` = negro) |
| `stroking_color` | Color del contorno |
| `ncs` | Espacio de color (`'DeviceGray'`, `'DeviceRGB'`…) |

**Estructura** (solo PDF etiquetados): `tag`, `mcid` → `None` en los exámenes.

> Para el clasificador basta con `text`, `size`, `fontname`, `x0`/`x1` y `top`/`bottom`.

In [ ]:
def left_neighbor(char, chars_by_x1):
    """Carácter que termina justo donde empieza `char` (a menos de 1,5 pt) y a su misma altura."""
    x = round(char["x0"])
    candidates = [other for key in (x - 1, x, x + 1) for other in chars_by_x1.get(key, [])
                  if other is not char
                  and abs(other["x1"] - char["x0"]) < 1.5
                  and other["top"] < char["bottom"] and char["top"] < other["bottom"]]
    return min(candidates, key=lambda other: abs(other["x1"] - char["x0"]), default=None)


def extract_page_text(page):
    """Texto de la página convirtiendo los sub/superíndices en caracteres Unicode."""
    # índice de los caracteres por su borde derecho (redondeado) para encontrar vecinos rápido
    chars_by_x1 = {}
    for char in page.chars:
        chars_by_x1.setdefault(round(char["x1"]), []).append(char)

    base_of = {}       # id(índice) -> carácter normal al que acompaña (la "H" de "H₂")
    characters = []
    for char in sorted(page.chars, key=lambda char: char["x0"]):   # de izquierda a derecha
        left = left_neighbor(char, chars_by_x1)
        # si el vecino ya es un índice (el "¹" de "10⁻¹⁴"), la base es la de ese vecino
        base = base_of.get(id(left), left)
        if base is not None and char["size"] < 0.9 * base["size"]:
            # superíndice si su base queda claramente por encima de la del carácter normal
            table = SUPERSCRIPT if char["bottom"] < base["bottom"] - 2 else SUBSCRIPT
            base_of[id(char)] = base
            # se le da la altura de su base para que extract_text lo ponga en la misma línea
            char = dict(char, text=char["text"].translate(table), top=base["top"], bottom=base["bottom"])
        characters.append(char)
    # extract_text junta en una línea los caracteres cuyo top difiere como máximo y_tolerance
    # y dentro de cada línea los ordena por x0
    return extract_text(characters, y_tolerance=4)

In [ ]:
# Primera línea de la cabecera que se repite en cada página (termina en la línea que contiene "CURSO")
HEADER_START = re.compile(r"UNIVERSIDADES DE ANDALUC|PRUEBA DE (ACCESO|EVALUACI)")
# "OPCIÓN A" (2010-2019) o "BLOQUE A" (2020-2024)
SECTION = re.compile(r"(?:OPCI[ÓO]N|BLOQUE) ([A-Z])\b")
# Inicio de pregunta: "1.-" (2010-2019), "PREGUNTA 1.-" (2025-) o "A1." (2020-2024)
QUESTION = re.compile(r"(?:PREGUNTA )?(\d+)\s*\.\s*[-–]|([A-Z])(\d+)\.\s")


def split_questions_from_file(path):
    """Devuelve una lista de preguntas: dicts con "section", "number" y "text"."""
    with pdfplumber.open(path) as pdf:
        text = "\n".join(extract_page_text(page) for page in pdf.pages)

    questions = []
    section = None     # letra de la OPCIÓN/BLOQUE actual (None si el examen no tiene)
    current = None     # pregunta a la que se añaden las líneas (None = descartar líneas)
    in_header = False
    for line in text.splitlines():
        if HEADER_START.match(line):           # empieza la cabecera: se salta...
            in_header = True
        elif in_header:                        # ...hasta la línea "CURSO 20XX-20XX" incluida
            in_header = "CURSO" not in line
        elif line.startswith("Instruccion"):   # instrucciones: se descartan hasta la siguiente pregunta
            current = None
        elif match := SECTION.search(line):    # nueva opción/bloque
            section, current = match[1], None
        elif match := QUESTION.match(line):    # nueva pregunta
            current = {"section": section or match[2],
                       "number": int(match[1] or match[3]),
                       "text": line}
            questions.append(current)
        elif current:                          # continuación de la pregunta actual
            current["text"] += "\n" + line
    return questions

In [19]:
file = ROOT / "examenes_quimica/2017_quimica/2017 - quimica - titular septiembre 6.pdf"

In [26]:
questions = split_questions_from_file(file)

In [28]:
questions

[]

In [27]:
len(questions)

0

In [17]:
for q in questions:
    print(f"[{q['section'] or ''}{q['number']}]", q["text"], sep="\n", end="\n" + "-" * 10 + "\n\n")